# Cell 1 — Import library

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    make_scorer
)

# Cell 2 — Tentukan folder project

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)
print("Output models dir:", OUTPUT_MODELS_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables
Output models dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\models


# Cell 3 — Baca dataset train/test dan feature set

In [3]:
TRAIN_DATASET_PATH = PROCESSED_DIR / "10_train_dataset.csv"
TEST_DATASET_PATH = PROCESSED_DIR / "10_test_dataset.csv"
FEATURE_SET_JSON_PATH = PROCESSED_DIR / "10_feature_sets.json"

train_df = pd.read_csv(TRAIN_DATASET_PATH)
test_df = pd.read_csv(TEST_DATASET_PATH)

with open(FEATURE_SET_JSON_PATH, "r", encoding="utf-8") as f:
    feature_sets = json.load(f)

target_col = "label_performance"

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("\nDistribusi train:")
display(train_df[target_col].value_counts())

print("\nDistribusi test:")
display(test_df[target_col].value_counts())

print("\nFeature sets:")
for name, features in feature_sets.items():
    print(name, "=", len(features), "fitur")

Train: (71, 49)
Test: (18, 49)

Distribusi train:


label_performance
Sedang    43
Tinggi    16
Rendah    12
Name: count, dtype: int64


Distribusi test:


label_performance
Sedang    11
Tinggi     4
Rendah     3
Name: count, dtype: int64


Feature sets:
all_process_features = 47 fitur
core_behavior_features = 20 fitur
activity_specific_features = 13 fitur
transition_specific_features = 14 fitur
frequency_and_ratio_features = 10 fitur


# Cell 4 — Baca hasil evaluasi Notebook 11 sebagai pembanding

In [4]:
PREVIOUS_EVALUATION_PATH = OUTPUT_TABLES_DIR / "11_model_evaluation_results.csv"

previous_results = pd.read_csv(PREVIOUS_EVALUATION_PATH)

print("Top 5 hasil Notebook 11:")
display(previous_results.head(5))

Top 5 hasil Notebook 11:


,feature_set,model,jumlah_fitur,cv_accuracy_mean,cv_accuracy_std,cv_macro_precision_mean,cv_macro_recall_mean,cv_macro_f1_mean,cv_macro_f1_std,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,all_process_features,Random_Forest_Balanced,47,0.620952,0.149424,0.567619,0.607407,0.554585,0.177150,0.722222,0.750000,0.848485,0.743231
1,core_behavior_features,Random_Forest_Balanced,20,0.494286,0.070912,0.427381,0.515741,0.436835,0.133236,0.722222,0.691667,0.795455,0.717836
2,transition_specific_features,Logistic_Regression_Balanced,14,0.480000,0.087877,0.392593,0.494444,0.417747,0.082420,0.666667,0.652381,0.765152,0.672222
3,frequency_and_ratio_features,Random_Forest_Balanced,10,0.480952,0.111066,0.435224,0.483333,0.435507,0.142265,0.666667,0.652381,0.765152,0.672222
4,activity_specific_features,Random_Forest_Balanced,13,0.649524,0.112066,0.588889,0.638889,0.594043,0.163538,0.611111,0.652778,0.734848,0.648459


# Cell 5 — Tentukan scorer dan cross-validation

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": make_scorer(accuracy_score),
    "macro_precision": make_scorer(
        precision_score,
        average="macro",
        zero_division=0
    ),
    "macro_recall": make_scorer(
        recall_score,
        average="macro",
        zero_division=0
    ),
    "macro_f1": make_scorer(
        f1_score,
        average="macro",
        zero_division=0
    )
}

print("Scoring utama: macro_f1")

Scoring utama: macro_f1


# Cell 6 — Tentukan model dan parameter grid

In [10]:
optimization_configs = {
    "Logistic_Regression_Tuned": {
        "model": Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        "param_grid": {
            "model__C": [0.01, 0.1, 1, 10],
            "model__solver": ["lbfgs"]
        }
    },

    "Random_Forest_Tuned": {
        "model": RandomForestClassifier(
            class_weight="balanced",
            random_state=42
        ),
        "param_grid": {
            "n_estimators": [100, 300, 500],
            "max_depth": [3, 5, None],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4],
            "max_features": ["sqrt", "log2"]
        }
    }
}

for model_name, config in optimization_configs.items():
    print(model_name)
    print(config["param_grid"])
    print()

Logistic_Regression_Tuned
{'model__C': [0.01, 0.1, 1, 10], 'model__solver': ['lbfgs']}

Random_Forest_Tuned
{'n_estimators': [100, 300, 500], 'max_depth': [3, 5, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4], 'max_features': ['sqrt', 'log2']}



# Cell 7 — Pilih feature set yang dituning

In [11]:
feature_sets_to_optimize = [
    "all_process_features",
    "core_behavior_features",
    "activity_specific_features",
    "transition_specific_features",
    "frequency_and_ratio_features"
]

for fs in feature_sets_to_optimize:
    print(fs, "=", len(feature_sets[fs]), "fitur")

all_process_features = 47 fitur
core_behavior_features = 20 fitur
activity_specific_features = 13 fitur
transition_specific_features = 14 fitur
frequency_and_ratio_features = 10 fitur


# Cell 8 — Fungsi evaluasi final di test set

In [12]:
def evaluate_on_test(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_macro_precision": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_recall": recall_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        ),
        "test_macro_f1": f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

    return metrics, y_pred

# Cell 9 — Jalankan Grid Search

In [13]:
optimization_rows = []
optimized_models = {}
optimized_predictions = {}

for feature_set_name in feature_sets_to_optimize:
    features = feature_sets[feature_set_name]

    X_train = train_df[features]
    y_train = train_df[target_col]

    X_test = test_df[features]
    y_test = test_df[target_col]

    print("\n==============================")
    print("Feature set:", feature_set_name)
    print("Jumlah fitur:", len(features))

    for model_name, config in optimization_configs.items():
        print("\nTuning:", model_name)

        grid_search = GridSearchCV(
            estimator=config["model"],
            param_grid=config["param_grid"],
            scoring=scoring,
            refit="macro_f1",
            cv=cv,
            n_jobs=-1,
            return_train_score=True,
            error_score="raise"
        )

        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        test_metrics, y_pred = evaluate_on_test(best_model, X_test, y_test)

        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "jumlah_fitur": len(features),
            "best_params": grid_search.best_params_,
            "cv_best_macro_f1": grid_search.best_score_,
            "cv_best_accuracy": grid_search.cv_results_["mean_test_accuracy"][grid_search.best_index_],
            "cv_best_macro_precision": grid_search.cv_results_["mean_test_macro_precision"][grid_search.best_index_],
            "cv_best_macro_recall": grid_search.cv_results_["mean_test_macro_recall"][grid_search.best_index_],
            **test_metrics
        }

        optimization_rows.append(row)

        key = f"{feature_set_name}__{model_name}"
        optimized_models[key] = best_model
        optimized_predictions[key] = {
            "y_test": y_test,
            "y_pred": y_pred,
            "features": features
        }

        print("Best params:", grid_search.best_params_)
        print("CV best macro F1:", round(grid_search.best_score_, 4))
        print("Test macro F1:", round(test_metrics["test_macro_f1"], 4))

optimization_results = pd.DataFrame(optimization_rows)

optimization_results = optimization_results.sort_values(
    by=["cv_best_macro_f1", "test_macro_f1"],
    ascending=False
).reset_index(drop=True)

display(optimization_results)


Feature set: all_process_features
Jumlah fitur: 47

Tuning: Logistic_Regression_Tuned
Best params: {'model__C': 0.01, 'model__solver': 'lbfgs'}
CV best macro F1: 0.4825
Test macro F1: 0.5667

Tuning: Random_Forest_Tuned
Best params: {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}
CV best macro F1: 0.6031
Test macro F1: 0.6485

Feature set: core_behavior_features
Jumlah fitur: 20

Tuning: Logistic_Regression_Tuned
Best params: {'model__C': 0.01, 'model__solver': 'lbfgs'}
CV best macro F1: 0.4825
Test macro F1: 0.5152

Tuning: Random_Forest_Tuned
Best params: {'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 300}
CV best macro F1: 0.5184
Test macro F1: 0.6722

Feature set: activity_specific_features
Jumlah fitur: 13

Tuning: Logistic_Regression_Tuned
Best params: {'model__C': 0.1, 'model__solver': 'lbfgs'}
CV best macro F1: 0.4575
Test macro F1: 0.6279

Tuning: Random_Fo

,feature_set,model,jumlah_fitur,best_params,cv_best_macro_f1,cv_best_accuracy,cv_best_macro_precision,cv_best_macro_recall,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,activity_specific_features,Random_Forest_Tuned,13,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.638170,0.693333,0.638030,0.671296,0.611111,0.620635,0.734848,0.627897
1,all_process_features,Random_Forest_Tuned,47,"{'max_depth': None, 'max_features': 'log2', 'm...",0.603123,0.678095,0.596111,0.648148,0.611111,0.652778,0.734848,0.648459
2,transition_specific_features,Random_Forest_Tuned,14,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.598652,0.648571,0.597057,0.671296,0.611111,0.681481,0.787879,0.632906
3,frequency_and_ratio_features,Random_Forest_Tuned,10,"{'max_depth': 5, 'max_features': 'sqrt', 'min_...",0.521342,0.579048,0.536111,0.586111,0.666667,0.652381,0.765152,0.672222
4,core_behavior_features,Random_Forest_Tuned,20,"{'max_depth': 3, 'max_features': 'sqrt', 'min_...",0.518450,0.579048,0.512405,0.600926,0.666667,0.652381,0.765152,0.672222
5,frequency_and_ratio_features,Logistic_Regression_Tuned,10,"{'model__C': 1, 'model__solver': 'lbfgs'}",0.508052,0.551429,0.526852,0.570370,0.555556,0.576190,0.704545,0.566667
6,transition_specific_features,Logistic_Regression_Tuned,14,"{'model__C': 0.01, 'model__solver': 'lbfgs'}",0.484389,0.522857,0.504141,0.572222,0.555556,0.576190,0.704545,0.570707
7,core_behavior_features,Logistic_Regression_Tuned,20,"{'model__C': 0.01, 'model__solver': 'lbfgs'}",0.482532,0.521905,0.492222,0.569444,0.500000,0.535714,0.674242,0.515152
8,all_process_features,Logistic_Regression_Tuned,47,"{'model__C': 0.01, 'model__solver': 'lbfgs'}",0.482456,0.536190,0.475397,0.554630,0.555556,0.576190,0.704545,0.566667
9,activity_specific_features,Logistic_Regression_Tuned,13,"{'model__C': 0.1, 'model__solver': 'lbfgs'}",0.457454,0.507619,0.461111,0.529630,0.611111,0.620635,0.734848,0.627897


# Cell 10 — Simpan hasil optimasi

In [14]:
OPTIMIZATION_RESULTS_PATH = OUTPUT_TABLES_DIR / "12_model_optimization_results.csv"

optimization_results.to_csv(OPTIMIZATION_RESULTS_PATH, index=False)

print("Hasil optimasi model disimpan ke:")
print(OPTIMIZATION_RESULTS_PATH)

Hasil optimasi model disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_model_optimization_results.csv


# Cell 11 — Pilih model terbaik berdasarkan CV macro F1

In [15]:
best_optimized_row = optimization_results.iloc[0]

best_optimized_feature_set = best_optimized_row["feature_set"]
best_optimized_model_name = best_optimized_row["model"]
best_optimized_key = f"{best_optimized_feature_set}__{best_optimized_model_name}"

print("Model terbaik berdasarkan CV macro F1:")
display(best_optimized_row)

print("\nBest optimized key:")
print(best_optimized_key)

Model terbaik berdasarkan CV macro F1:


feature_set                                       activity_specific_features
model                                                    Random_Forest_Tuned
jumlah_fitur                                                              13
best_params                {'max_depth': None, 'max_features': 'sqrt', 'm...
cv_best_macro_f1                                                     0.63817
cv_best_accuracy                                                    0.693333
cv_best_macro_precision                                              0.63803
cv_best_macro_recall                                                0.671296
test_accuracy                                                       0.611111
test_macro_precision                                                0.620635
test_macro_recall                                                   0.734848
test_macro_f1                                                       0.627897
Name: 0, dtype: object


Best optimized key:
activity_specific_features__Random_Forest_Tuned


# Cell 12 — Classification report model optimized terbaik

In [16]:
best_opt_predictions = optimized_predictions[best_optimized_key]

y_test_best_opt = best_opt_predictions["y_test"]
y_pred_best_opt = best_opt_predictions["y_pred"]

print("Classification report model optimized terbaik:")
print(classification_report(
    y_test_best_opt,
    y_pred_best_opt,
    zero_division=0
))

Classification report model optimized terbaik:
              precision    recall  f1-score   support

      Rendah       0.60      1.00      0.75         3
      Sedang       0.83      0.45      0.59        11
      Tinggi       0.43      0.75      0.55         4

    accuracy                           0.61        18
   macro avg       0.62      0.73      0.63        18
weighted avg       0.70      0.61      0.61        18



# Cell 13 — Confusion matrix model optimized terbaik

In [17]:
labels_order = ["Rendah", "Sedang", "Tinggi"]

cm_opt = confusion_matrix(
    y_test_best_opt,
    y_pred_best_opt,
    labels=labels_order
)

confusion_matrix_opt_df = pd.DataFrame(
    cm_opt,
    index=[f"Actual_{label}" for label in labels_order],
    columns=[f"Predicted_{label}" for label in labels_order]
)

display(confusion_matrix_opt_df)

CONFUSION_MATRIX_OPT_PATH = OUTPUT_TABLES_DIR / "12_best_optimized_model_confusion_matrix.csv"
confusion_matrix_opt_df.to_csv(CONFUSION_MATRIX_OPT_PATH)

print("Confusion matrix optimized model disimpan ke:")
print(CONFUSION_MATRIX_OPT_PATH)

,Predicted_Rendah,Predicted_Sedang,Predicted_Tinggi
Actual_Rendah,3,0,0
Actual_Sedang,2,5,4
Actual_Tinggi,0,1,3


Confusion matrix optimized model disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_best_optimized_model_confusion_matrix.csv


# Cell 14 — Simpan classification report optimized model

In [18]:
report_opt_dict = classification_report(
    y_test_best_opt,
    y_pred_best_opt,
    zero_division=0,
    output_dict=True
)

classification_report_opt_df = pd.DataFrame(report_opt_dict).T

CLASSIFICATION_REPORT_OPT_PATH = OUTPUT_TABLES_DIR / "12_best_optimized_model_classification_report.csv"
classification_report_opt_df.to_csv(CLASSIFICATION_REPORT_OPT_PATH)

display(classification_report_opt_df)

print("Classification report optimized model disimpan ke:")
print(CLASSIFICATION_REPORT_OPT_PATH)

,precision,recall,f1-score,support
Rendah,0.600000,1.000000,0.750000,3.000000
Sedang,0.833333,0.454545,0.588235,11.000000
Tinggi,0.428571,0.750000,0.545455,4.000000
accuracy,0.611111,0.611111,0.611111,0.611111
macro avg,0.620635,0.734848,0.627897,18.000000
weighted avg,0.704497,0.611111,0.605689,18.000000


Classification report optimized model disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_best_optimized_model_classification_report.csv


# Cell 15 — Bandingkan model terbaik Notebook 11 vs Notebook 12

In [19]:
previous_best = previous_results.iloc[0]

comparison_best_models = pd.DataFrame([
    {
        "source": "Notebook 11 - Untuned",
        "feature_set": previous_best["feature_set"],
        "model": previous_best["model"],
        "jumlah_fitur": previous_best["jumlah_fitur"],
        "cv_macro_f1": previous_best["cv_macro_f1_mean"],
        "test_accuracy": previous_best["test_accuracy"],
        "test_macro_precision": previous_best["test_macro_precision"],
        "test_macro_recall": previous_best["test_macro_recall"],
        "test_macro_f1": previous_best["test_macro_f1"]
    },
    {
        "source": "Notebook 12 - Optimized",
        "feature_set": best_optimized_row["feature_set"],
        "model": best_optimized_row["model"],
        "jumlah_fitur": best_optimized_row["jumlah_fitur"],
        "cv_macro_f1": best_optimized_row["cv_best_macro_f1"],
        "test_accuracy": best_optimized_row["test_accuracy"],
        "test_macro_precision": best_optimized_row["test_macro_precision"],
        "test_macro_recall": best_optimized_row["test_macro_recall"],
        "test_macro_f1": best_optimized_row["test_macro_f1"]
    }
])

display(comparison_best_models)

COMPARISON_BEST_MODELS_PATH = OUTPUT_TABLES_DIR / "12_comparison_untuned_vs_optimized.csv"
comparison_best_models.to_csv(COMPARISON_BEST_MODELS_PATH, index=False)

print("Perbandingan model terbaik disimpan ke:")
print(COMPARISON_BEST_MODELS_PATH)

,source,feature_set,model,jumlah_fitur,cv_macro_f1,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,Notebook 11 - Untuned,all_process_features,Random_Forest_Balanced,47,0.554585,0.722222,0.750000,0.848485,0.743231
1,Notebook 12 - Optimized,activity_specific_features,Random_Forest_Tuned,13,0.638170,0.611111,0.620635,0.734848,0.627897


Perbandingan model terbaik disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_comparison_untuned_vs_optimized.csv


# Cell 16 — Feature importance model optimized terbaik

In [20]:
best_optimized_model = optimized_models[best_optimized_key]
best_optimized_features = optimized_predictions[best_optimized_key]["features"]

if best_optimized_model_name == "Random_Forest_Tuned":
    feature_importance_opt_df = pd.DataFrame({
        "feature": best_optimized_features,
        "importance": best_optimized_model.feature_importances_
    }).sort_values("importance", ascending=False)

    display(feature_importance_opt_df.head(20))

    FEATURE_IMPORTANCE_OPT_PATH = OUTPUT_TABLES_DIR / "12_best_optimized_feature_importance.csv"
    feature_importance_opt_df.to_csv(FEATURE_IMPORTANCE_OPT_PATH, index=False)

    print("Feature importance optimized model disimpan ke:")
    print(FEATURE_IMPORTANCE_OPT_PATH)

else:
    print("Model optimized terbaik bukan Random Forest.")

,feature,importance
7,act_quiz_auto_saved,0.173160
5,act_quiz_viewed,0.134892
6,act_quiz_updated,0.105484
8,act_quiz_summary_viewed,0.085161
1,act_material_viewed,0.078630
0,act_course_viewed,0.068313
3,act_quiz_access_prevented,0.060054
9,act_quiz_submitted,0.056880
2,act_quiz_module_viewed,0.051748
11,act_assignment_status_viewed,0.051142


Feature importance optimized model disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_best_optimized_feature_importance.csv


# Cell 17 — Koefisien Logistic Regression optimized terbaik

In [21]:
if best_optimized_model_name == "Logistic_Regression_Tuned":
    logistic_model = best_optimized_model.named_steps["model"]

    coef_opt_df = pd.DataFrame(
        logistic_model.coef_,
        columns=best_optimized_features,
        index=logistic_model.classes_
    )

    display(coef_opt_df)

    COEF_OPT_PATH = OUTPUT_TABLES_DIR / "12_best_optimized_logistic_coefficients.csv"
    coef_opt_df.to_csv(COEF_OPT_PATH)

    print("Koefisien Logistic Regression optimized disimpan ke:")
    print(COEF_OPT_PATH)

else:
    print("Model optimized terbaik bukan Logistic Regression.")

Model optimized terbaik bukan Logistic Regression.


# Cell 18 — Simpan ringkasan model optimized terbaik

In [22]:
best_optimized_summary = pd.DataFrame([{
    "best_feature_set": best_optimized_feature_set,
    "best_model": best_optimized_model_name,
    "jumlah_fitur": int(best_optimized_row["jumlah_fitur"]),
    "best_params": str(best_optimized_row["best_params"]),
    "cv_best_macro_f1": best_optimized_row["cv_best_macro_f1"],
    "test_accuracy": best_optimized_row["test_accuracy"],
    "test_macro_precision": best_optimized_row["test_macro_precision"],
    "test_macro_recall": best_optimized_row["test_macro_recall"],
    "test_macro_f1": best_optimized_row["test_macro_f1"]
}])

display(best_optimized_summary)

BEST_OPTIMIZED_SUMMARY_PATH = OUTPUT_TABLES_DIR / "12_best_optimized_model_summary.csv"
best_optimized_summary.to_csv(BEST_OPTIMIZED_SUMMARY_PATH, index=False)

print("Ringkasan model optimized terbaik disimpan ke:")
print(BEST_OPTIMIZED_SUMMARY_PATH)

,best_feature_set,best_model,jumlah_fitur,best_params,cv_best_macro_f1,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,activity_specific_features,Random_Forest_Tuned,13,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.63817,0.611111,0.620635,0.734848,0.627897


Ringkasan model optimized terbaik disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\12_best_optimized_model_summary.csv


## Kesimpulan Model Optimization

Pada tahap ini, dilakukan optimasi hyperparameter terhadap model Logistic Regression dan Random Forest menggunakan beberapa feature set hasil process mining. Optimasi dilakukan menggunakan GridSearchCV dengan stratified 5-fold cross-validation pada data latih. Metrik utama yang digunakan untuk pemilihan model terbaik adalah macro F1-score karena target terdiri dari tiga kelas dengan distribusi yang tidak seimbang.

Hasil optimasi menunjukkan bahwa model terbaik berdasarkan nilai cross-validation macro F1 adalah Random Forest Tuned dengan feature set `activity_specific_features`. Model ini menggunakan 13 fitur aktivitas spesifik dan memperoleh CV macro F1-score sebesar 0,6382. Pada data uji, model ini memperoleh akurasi sebesar 61,11% dan macro F1-score sebesar 0,6279.

Jika dibandingkan dengan model terbaik pada tahap sebelumnya, yaitu Random Forest Balanced dengan `all_process_features`, model hasil optimasi memiliki nilai cross-validation macro F1 yang lebih tinggi, tetapi nilai macro F1 pada data uji lebih rendah. Model terbaik pada tahap sebelumnya memperoleh test macro F1-score sebesar 0,7432, sedangkan model hasil optimasi memperoleh test macro F1-score sebesar 0,6279. Perbedaan ini menunjukkan bahwa hasil evaluasi masih cukup dipengaruhi oleh ukuran data yang terbatas, terutama karena data uji hanya terdiri dari 18 mahasiswa.

Berdasarkan confusion matrix, model optimized mampu mengklasifikasikan seluruh mahasiswa pada kelas Rendah dengan benar, tetapi masih mengalami kesulitan dalam membedakan kelas Sedang dan Tinggi. Sebagian mahasiswa dengan label Sedang diprediksi sebagai Tinggi, dan sebagian mahasiswa Tinggi diprediksi sebagai Sedang. Hal ini menunjukkan bahwa pola aktivitas mahasiswa pada kelas Sedang dan Tinggi memiliki kemiripan tertentu.

Feature importance pada model optimized menunjukkan bahwa fitur aktivitas kuis menjadi fitur yang paling dominan, seperti `act_quiz_auto_saved`, `act_quiz_viewed`, `act_quiz_updated`, dan `act_quiz_summary_viewed`. Selain itu, fitur akses material dan course juga muncul sebagai fitur penting. Temuan ini konsisten dengan hasil process mining sebelumnya yang menunjukkan bahwa proses belajar mahasiswa pada LMS didominasi oleh aktivitas kuis, dengan dukungan aktivitas akses course dan material.

Dengan demikian, optimasi hyperparameter memberikan gambaran bahwa fitur aktivitas spesifik memiliki performa cross-validation yang paling stabil. Namun, model dengan performa data uji tertinggi tetap diperoleh pada tahap sebelumnya menggunakan Random Forest Balanced dengan seluruh fitur process mining. Oleh karena itu, hasil akhir perlu diinterpretasikan secara hati-hati dan tidak dianggap sebagai model prediksi final yang mutlak, melainkan sebagai analisis pendukung untuk memahami hubungan antara pola aktivitas LMS dan performa akademik mahasiswa.